# 02 · Fine-tune Qwen2.5-Coder (QLoRA) for Text2SQL — Colab T4 / Kaggle

Free-tier recipe: **Unsloth + QLoRA** on `Qwen2.5-Coder-1.5B-Instruct`
(Apache-2.0, 1.54B params). Designed to fit **Kaggle's 19 GB** working disk.

**Key storage trick:** training never executes SQL, so it only needs the
*schema text* — we build that from BIRD's small `train_tables.json` and **skip
the ~8 GB `train_databases` download entirely**. Only the ~1.3 GB
`dev_databases` is fetched, for execution-accuracy eval.

> **Colab:** Runtime → Change runtime type → **T4 GPU**.  
> **Kaggle:** Settings → Accelerator → **GPU T4 x2**, and **Internet → On**.  
> Then run all cells.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(enable the GPU runtime!)')

## 1. Get the code (clone the repo)

In [ ]:
# --- SETUP: run me first (works on Colab, Kaggle AND locally) ---
import os, sys, subprocess
if not os.path.exists('src') and os.path.basename(os.getcwd()) != 'text2sql-finetuning':
    if not os.path.isdir('text2sql-finetuning'):
        subprocess.run(['git', 'clone', 'https://github.com/Shiverion/text2sql-finetuning.git'], check=True)
    os.chdir('text2sql-finetuning')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src present :', os.path.isdir('src'))

## 2. Install Unsloth + training stack
`pip install unsloth` pulls a torch/transformers/trl/peft combo tested
together — don't hand-pin these unless you know why. (~2–4 min.)

In [ ]:
!pip install -q unsloth
# If you hit a version error later, get the matching nightly instead:
# !pip install -q --upgrade --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git" \
#     "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"

## 3. Download BIRD — storage-lean (fits 19 GB)
Big zips go to `/tmp` (scratch, **not** counted against Kaggle's 19 GB working
quota). From `train.zip` we extract **only** `train.json` + `train_tables.json`
(a few MB) and skip the ~8 GB databases. For dev we extract the real `.sqlite`
(needed to *execute* queries). Zips are deleted right after extraction.

In [ ]:
import os, glob
SCRATCH = '/tmp/bird_dl'; os.makedirs(SCRATCH, exist_ok=True)
REPO = os.getcwd()
BIRD = 'https://bird-bench.oss-cn-beijing.aliyuncs.com'

# --- TRAIN: schemas only (no 8 GB databases) ---
if not os.path.exists('train/train_tables.json'):
    !wget -q -c {BIRD}/train.zip -O {SCRATCH}/train.zip
    !unzip -o {SCRATCH}/train.zip 'train/train.json' 'train/train_tables.json' -d {REPO}
    !rm -f {SCRATCH}/train.zip   # free ~8.3 GB immediately

# --- DEV: real databases for execution eval (~1.3 GB) ---
if not glob.glob('dev*/dev.json'):
    !wget -q -c {BIRD}/dev.zip -O {SCRATCH}/dev.zip
    !unzip -o {SCRATCH}/dev.zip -d {REPO}
    !rm -f {SCRATCH}/dev.zip
for z in glob.glob('dev*/dev_databases.zip'):   # un-nest the dev DBs
    !unzip -o '{z}' -d '{os.path.dirname(z)}'
    !rm -f '{z}'

print('--- disk usage ---')
!du -sh train dev* data/processed 2>/dev/null; df -h {REPO} | tail -1

### Resolve paths
Auto-detects the dev folder name (BIRD has shipped it as `dev/` and
`dev_20240627/` in different releases).

In [ ]:
TRAIN_JSON   = 'train/train.json'
TRAIN_TABLES = 'train/train_tables.json'   # schemas WITHOUT databases
dev_jsons = glob.glob('dev*/dev.json')
DEV_DIR  = os.path.dirname(dev_jsons[0]) if dev_jsons else 'dev'
DEV_JSON = f'{DEV_DIR}/dev.json'
DEV_DB   = f'{DEV_DIR}/dev_databases'      # real .sqlite for execution eval
print('train :', TRAIN_JSON, '+', TRAIN_TABLES)
print('dev   :', DEV_JSON, '| db_root:', DEV_DB, '| exists:', os.path.isdir(DEV_DB))

## 4. Preprocess → JSONL
Train uses `--tables_json` (schemas, no databases). Dev uses `--db_root` so
the predicted SQL can later be executed against the real databases.

In [ ]:
!python -m src.data_prep --source bird --json {TRAIN_JSON} --tables_json {TRAIN_TABLES} \
    --out data/processed/bird_train.jsonl --shuffle
!python -m src.data_prep --source bird --json {DEV_JSON} --db_root {DEV_DB} \
    --out data/processed/bird_dev.jsonl

In [ ]:
# sanity: both files should be non-empty (dev > 0 confirms the DBs extracted)
import json
for f in ['data/processed/bird_train.jsonl', 'data/processed/bird_dev.jsonl']:
    n = sum(1 for _ in open(f, encoding='utf-8'))
    print(f'{f}: {n} records')
print()
print(json.dumps(json.loads(open('data/processed/bird_train.jsonl').readline()), indent=2)[:1000])

## 5. Train (QLoRA)
Runs the hardened trainer in `src/train.py` (Unsloth + completion-only loss;
SFTConfig/SFTTrainer kwargs auto-adapt to the installed TRL version).

`--max_train_samples 8000 --epochs 2` ≈ 900 steps ≈ ~2.5 h on a T4. For a quick
first pass use `--max_steps 60` (a few minutes) to confirm everything runs.

In [ ]:
!python -m src.train --preset exp1_qwen1.5b_bird_qlora \
    --train_file data/processed/bird_train.jsonl \
    --val_file   data/processed/bird_dev.jsonl \
    --max_train_samples 8000 --epochs 2

Adapter + tokenizer are saved to `outputs/qwen2.5-coder-1.5b-bird-qlora/`
(a few MB — well within 19 GB). Implementation: [`src/train.py`](src/train.py);
hyper-parameters: the presets in [`src/config.py`](src/config.py).

## 6. (Optional) push the adapter to the Hugging Face Hub
Set an `HF_TOKEN` secret first — **Kaggle:** Add-ons → Secrets → `HF_TOKEN`;
**Colab:** key icon → `HF_TOKEN`.

In [ ]:
import os
from huggingface_hub import HfApi, login
ADAPTER_DIR = 'outputs/qwen2.5-coder-1.5b-bird-qlora'
HF_REPO     = 'Shiverion/qwen2.5-coder-1.5b-bird-qlora'  # <- change to your username

token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception: pass
if token is None:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception: pass

if token and os.path.isdir(ADAPTER_DIR):
    login(token=token)
    api = HfApi(); api.create_repo(HF_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=HF_REPO, repo_type='model')
    print('pushed ->', f'https://huggingface.co/{HF_REPO}')
else:
    print('skipped: set the HF_TOKEN secret and make sure', ADAPTER_DIR, 'exists')

Now run **03_inference_eval.ipynb** (same session) to measure execution accuracy.